# Architecture Overview: LangChain Data Analysis Agent

This document explains the components and architectural flow implemented in the Python notebook [Langchain_data_analysis_from_1st_principles.ipynb](file:///Users/nitinaggarwal/Documents/learning/langgraph_deep_agents/learned_stuff/Learn_projects_from_LangChain/Langchain_data_analysis_from_1st_principles/Langchain_data_analysis_from_1st_principles.ipynb).

## Component Architecture & Interaction Diagram

The diagram below details how the parent Agent, its custom middlewares, and the isolated Daytona sandbox execution environment interact.

![Agent Architecture Diagram](data_analysis_agent_architecture_1782474406078.png)

---

## Detailed Component Flow

### 1. Parent Agent (`create_agent`)
- **Orchestration:** Orchestrated by the LangGraph engine to run a reasoning loop.
- **Model:** Communicates with the configured LLM (e.g., Llama 3 or Gemini) to parse user intents and execute tools.

### 2. Middleware Layer
The middleware intercepts and wraps agent execution to inject capabilities dynamically:
- **`FileSystemMiddleware`:**
  - Dynamic Tool Injection: Automatically exposes file operations (`read_file`, `write_file`, `ls`) as tools to the agent.
  - Path Scoping: Ensures files read/written by the agent map directly to the sandbox filesystem.
- **`SummarizationMiddleware`:**
  - Context Trimming: Monitors message tokens and dynamically condenses history to prevent context window overflow during longer conversations.
- **`SkillsMiddleware`:**
  - Domain-Specific Instruction: Injects pre-defined guidelines and patterns (such as pandas and matplotlib practices copied from local files) directly into the agent’s system instructions.
- **`SubAgentMiddleware`:**
  - Delegation: Spawns and manages specialized helper agents (like a `visualizer` subagent) to offload complex tasks (e.g., generating Matplotlib charts).

### 3. Execution Backend (`DaytonaSandbox`)
- **Isolation:** A secure, remote container/VM environment where code execution and file modifications occur safely.
- **Pre-seeding:** Stores datasets (like `sales_csv_data.csv`) and local skill definitions (like `pandas-patterns/SKILL.md`) uploaded before execution.


In [58]:
import os
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [66]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from deepagents.backends.langsmith import LangSmithSandbox

from daytona_sandbox import *
from deepagents.middleware import FilesystemMiddleware,SummarizationMiddleware,SkillsMiddleware,SubAgentMiddleware
from deepagents import SubAgent
from langchain.chat_models import init_chat_model
## from langsmith.sandbox import SandboxClient


## Clean previous stale sandboxes if new one is getting created
# cleanup_stale_sandboxes()
# backend = create_new_sandbox_with_daytona()

backend = connect_with_existing_sandbox_with_daytona(sandbox_id="58699b79-b36a-4f46-a5fc-3bb8af5169df")

llm = init_chat_model(
    model="google_genai:gemini-3.1-flash-lite",
    api_key= os.getenv("GOOGLE_API_KEY")
)

chat_summarization_model = "google_genai:gemini-3.1-flash-lite"

## Copying skills from local machine to the sandbox machine
upload_skill_response = upload_skills_to_daytona(
    backend=backend,
    src_skill_folder_path="/Users/nitinaggarwal/Documents/learning/langgraph_deep_agents/learned_stuff/Learn_projects_from_LangChain/Langchain_data_analysis_from_1st_principles/skills",
    sandbox_dest_path="/home/daytona//skills"
    )

### Pass the Sub-Agent
visualizer: SubAgent = {
    "name": "visualizer",
    "description": "Generates charts and visualizations from data files in the sandbox.",
    "system_prompt": "You are a data visualization specialist. Write Python scripts using matplotlib and seaborn. Save all figures as PNG files.",
    "tools": [],
    "model": "google_genai:gemini-3.1-flash-lite",
}


if upload_skill_response["status"]=="success":
    print(backend.id)
    agent = create_agent(
        model=llm,
        tools=[],
        middleware=[
            FilesystemMiddleware(backend=backend),
            SummarizationMiddleware(
                model=llm,
                backend=backend
                ),
            SkillsMiddleware(backend=backend, sources=["/home/daytona//skills/"]),
            TodoListMiddleware(),
            SubAgentMiddleware(backend=backend,subagents=[visualizer])
            ],
    )
else:
    print(upload_skill_response)
    create_sandbox_with_daytona()

58699b79-b36a-4f46-a5fc-3bb8af5169df


In [60]:
print(upload_skill_response)

{'status': 'success', 'uploaded_files': ['/home/daytona//skills/pandas-patterns/SKILL.md'], 'missing_files': []}


In [61]:
## Wtite some data to Sandbox

sales_csv_data_2026 = [
    ["Date", "Product", "Units", "Revenue"],
    ["2025-08-01", "Widget A", 10, 250],
    ["2025-08-02", "Widget B", 5, 125],
    ["2025-08-03", "Widget A", 7, 175],
    ["2025-08-04", "Widget C", 3, 990],
]


response = write_csv_to_sandbox(
    backend=backend,
    data=sales_csv_data,
    path="/home/daytona/sales_csv_data_2026.csv",
    overwrite=True
)

print(response)

{'status': 'success', 'path': '/home/daytona/sales_csv_data_2026.csv', 'rows_written': 5, 'lines_on_sandbox': 5, 'content_match': True, 'error': None}


In [62]:
from langchain.messages import HumanMessage
from utils import format_messages

In [67]:
data_analysis_prompt="""
Read "/home/daytona/sales_csv_data_2026.csv" and summarize total revenue by product in one sentence summary and MUST generate a bar chart image and save it at "/home/daytona/sales_csv_data_2026.png".
"""

In [68]:
## Stream agent response


result = agent.invoke(
    {
        "messages":[
            HumanMessage(
                content=data_analysis_prompt
            )
        ]
    },
    version="v3",
    config={"recursion_limit": 30},
)
format_messages(result["messages"])

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Read "/home/daytona/sales_csv_data_2026.csv" and summarize total revenue by product in one sentence summary and │
│ MUST generate a bar chart image and save it at "/home/daytona/sales_csv_data_2026.png".                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 🔧 Tool Call: ls                                                                                                │
│    Args: {                                                                                                      │
│   "path": "/home/daytona/"                                                                                      │
│ }                                                                                                               │
│    ID: d377lFWo                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ ['/home/daytona/.bash_logout', '/home/daytona/.bashrc', '/home/daytona/.face', '/home/daytona/.face.icon',      │
│ '/home/daytona/.profile', '/home/daytona/.daytona', '/home/daytona/skills', '/home/daytona/sales_csv_data.csv', │
│ '/home/daytona/.zcompdump', '/home/daytona/.config', '/home/daytona/data', '/home/daytona/.cache',              │
│ '/home/daytona/process_sales.py', '/home/daytona/sales_csv_data.png', '/home/daytona/generate_plot.py',         │
│ '/home/daytona/sales_csv_data_2026.csv', '/home/daytona/.zshrc']                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 🔧 Tool Call: read_file                                                                                         │
│    Args: {                                                                                                      │
│   "file_path": "/home/daytona/sales_csv_data_2026.csv",                                                         │
│   "limit": 10                                                                                                   │
│ }                                                                                                               │
│    ID: SU2S7aUN                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│      1  Date,Product,Units,Revenue                                                                              │
│      2  2025-08-01,Widget A,10,250                                                                              │
│      3  2025-08-02,Widget B,5,125                                                                               │
│      4  2025-08-03,Widget A,7,175                                                                               │
│      5  2025-08-04,Widget C,3,90                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 🔧 Tool Call: execute                                                                                           │
│    Args: {                                                                                                      │
│   "command": "python3 -c \"import pandas as pd; df = pd.read_csv('/home/daytona/sales_csv_data_2026.csv');      │
│ print(df.groupby('Product')['Revenue'].sum())\""                                                                │
│ }                                                                                                               │
│    ID: xe325wCj                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Product                                                                                                         │
│ Widget A    425                                                                                                 │
│ Widget B    125                                                                                                 │
│ Widget C     90                                                                                                 │
│ Name: Revenue, dtype: int64                                                                                     │
│                                                                                                                 │
│ [Command succeeded with exit code 0]                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 🔧 Tool Call: execute                                                                                           │
│    Args: {                                                                                                      │
│   "command": "python3 -c \"import pandas as pd; import seaborn as sns; import matplotlib.pyplot as plt; df =    │
│ pd.read_csv('/home/daytona/sales_csv_data_2026.csv'); product_revenue =                                         │
│ df.groupby('Product')['Revenue'].sum().sort_values(ascending=False).reset_index(); plt.figure(figsize=(10, 6)); │
│ sns.barplot(data=product_revenue, x='Product', y='Revenue', palette='viridis'); plt.title('Total Revenue by     │
│ Product (2026)'); plt.xlabel('Product'); plt.ylabel('Total Revenue'); plt.tight_layout();                       │
│ plt.savefig('/home/daytona/sales_csv_data_2026.png');\""                                                        │
│ }                                                                                                               │
│    ID: U4M75ecC                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│                                                                                                                 │
│ <stderr><string>:1: FutureWarning:                                                                              │
│                                                                                                                 │
│ Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable │
│ to `hue` and set `legend=False` for the same effect.</stderr>                                                   │
│ [Command succeeded with exit code 0]                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│ The total revenue generated by each product is 425 for Widget A, 125 for Widget B, and 90 for Widget C, making  │
│ Widget A the highest-earning product, and the bar chart illustrating these results has been saved at            │
│ /home/daytona/sales_csv_data_2026.png.                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
### Provide data analysis skills to the model